In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [3]:
RESULTS_CSV = "evaluation/local_run_backup.csv"  # update this path if needed

df = pd.read_csv(RESULTS_CSV)
print(f"Loaded {len(df)} runs")
df.head()

Loaded 336 runs


,run_name,run_id,data_version,include_food,include_rain,include_text,conflict_only_embeddings,use_pca,k,n_splits,...,param_min_child_weight,param_max_depth,param_max_delta_step,param_learning_rate,param_gamma,param_colsample_bytree,param_colsample_bylevel,param_k,param_n_splits,param_event_col
0,acled_sub_food_rain_text_conflict_pca_0.25_4,c0cf7ee2ad4a42e5bcb3f2d9f5db43d6,acled_sub_food_rain_text_conflict,True,True,True,True,True,0.25,4,...,5,7,5,0.03,3,0.6,0.8,0.25,4,sub_event_type
1,acled_sub_food_rain_text_all_pca_0.25_4,9168691cdd7a43e9a772411e1265a240,acled_sub_food_rain_text_all,True,True,True,False,True,0.25,4,...,1,3,0,0.10,1,0.6,0.6,0.25,4,sub_event_type
2,acled_event_food_rain_text_conflict_pca_0.25_4,caa2fef35ad543b9af54726579c98f18,acled_event_food_rain_text_conflict,True,True,True,True,True,0.25,4,...,5,3,0,0.10,3,1.0,0.8,0.25,4,event_type
3,acled_event_food_rain_text_all_pca_0.25_4,ec5b38251c944f0a980be616f72aa8bc,acled_event_food_rain_text_all,True,True,True,False,True,0.25,4,...,1,5,0,0.10,3,0.8,0.8,0.25,4,event_type
4,acled_sub_food_rain_text_conflict_pca_0.5_4,4ddbff406f124e11858ebf40b024af68,acled_sub_food_rain_text_conflict,True,True,True,True,True,0.50,4,...,5,7,5,0.10,5,0.6,0.6,0.50,4,sub_event_type


### K - escalation threshold
`k` controls the esclation threshold, a lower k = a looser threshold. 
Raw AUPR favours the lowest `k`, but that is misleading in terms of conflict prediction. 

In [4]:
collapsed = df[df["onset_recall_class1"] == 1.0]
baseline_by_k = collapsed.groupby("k")["onset_precision_class1"].median()

collapse_rate = df.groupby("k")["onset_recall_class1"].apply(lambda x: (x == 1.0).mean())

summary = pd.DataFrame({
    "mean_onset_aupr": df.groupby("k")["onset_aupr"].mean(),
    "max_onset_aupr": df.groupby("k")["onset_aupr"].max(),
    "best_non_collapsed_aupr": df[df["onset_recall_class1"] < 1.0].groupby("k")["onset_aupr"].max(),
    "baseline_prevalence": baseline_by_k,
    "collapse_rate": collapse_rate,
})
summary["lift_over_baseline"] = summary["best_non_collapsed_aupr"] / summary["baseline_prevalence"]
summary.round(3)

,mean_onset_aupr,max_onset_aupr,best_non_collapsed_aupr,baseline_prevalence,collapse_rate,lift_over_baseline
k,,,,,,
0.25,0.440,0.526,0.482,0.421,0.750,1.145
0.50,0.368,0.458,0.423,0.370,0.641,1.142
1.00,0.338,0.421,0.421,0.306,0.281,1.376
